# FINE-TUNNING FOR OPENVLA

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
# FINE-TUNNING SCRIPT FOR OPENVLA MODEL PART 1
# CONFIGURATIONS

import os
os.chdir("/home/ids/ext-5219/tokenizer/openvla-oft/")  # replace with your repo root
print("Current working directory:", os.getcwd())

sys.argv.append("pusht")

from collections import deque
from dataclasses import dataclass
from pathlib import Path
from typing import Optional

import draccus
import torch
import torch.distributed as dist
import tqdm
from accelerate import PartialState
from peft import LoraConfig, PeftModel, get_peft_model, prepare_model_for_kbit_training
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.optim import AdamW
from torch.utils.data import DataLoader
from transformers import AutoModelForVision2Seq, AutoProcessor, BitsAndBytesConfig
from transformers import AutoConfig, AutoImageProcessor
from transformers.modeling_outputs import CausalLMOutputWithPast

import wandb
from prismatic.models.backbones.llm.prompting import PurePromptBuilder, VicunaV15ChatPromptBuilder
from prismatic.util.data_utils import PaddedCollatorForActionPrediction
from prismatic.vla.action_tokenizer import ActionTokenizer
from prismatic.vla.datasets import RLDSBatchTransform, RLDSDataset
from prismatic.vla.datasets.rlds.utils.data_utils import save_dataset_statistics

from prismatic.extern.hf.configuration_prismatic import OpenVLAConfig
from prismatic.extern.hf.modeling_prismatic import OpenVLAForActionPrediction
from prismatic.extern.hf.processing_prismatic import PrismaticImageProcessor, PrismaticProcessor

# Sane Defaults
os.environ["TOKENIZERS_PARALLELISM"] = "false"


# # === Utilities ===
# # fmt: off
# def create_vision_transform(vla: nn.Module, input_size: int) -> Callable[[Image.Image], torch.Tensor]:
#     """Gets image transform for the vision encoder."""
#     data_cfg = timm.data.resolve_model_data_config(vla.vision_backbone)
#     data_cfg["input_size"] = (3, input_size, input_size)
#     return timm.data.create_transform(
#         input_size=data_cfg["input_size"],
#         interpolation=data_cfg["interpolation"],
#         mean=data_cfg["mean"],
#         std=data_cfg["std"],
#         crop_pct=1.0,           # Set to 1.0 to disable cropping
#         crop_mode="center",     # Default crop mode --> no-op when `crop_pct == 1.0`
#         is_training=False,      # Disable image_aug when loading transform; handled by RLDS dataloader
#     )
#
# # fmt: on



# fmt: off
vla_path: str = "openvla/openvla-7b"                            # Path to OpenVLA model (on HuggingFace Hub)

# Directory Paths
data_root_dir: Path = Path("/home/ids/ext-5219/tokenizer/test")        # Path to Open-X dataset directory
dataset_name: str = "columbia_cairlab_pusht_real"                                # Name of fine-tuning dataset (e.g., `droid_wipe`)
run_root_dir: Path = Path("runs")                               # Path to directory to store logs & checkpoints
adapter_tmp_dir: Path = Path("adapter-tmp")                     # Temporary directory for LoRA weights before fusing

# Fine-tuning Parameters
batch_size: int = 8                                            # Fine-tuning batch size
max_steps: int = 300                                        # Max number of fine-tuning steps
save_steps: int = 150                                          # Interval for checkpoint saving
learning_rate: float = 5e-4                                     # Fine-tuning learning rate
grad_accumulation_steps: int = 1                                # Gradient accumulation steps
image_aug: bool = True                                          # Whether to train with image augmentations
shuffle_buffer_size: int = 10000                              # Dataloader shuffle buffer size (can reduce if OOM)
save_latest_checkpoint_only: bool = True                        # Whether to save only one checkpoint per run and
                                                                #   continually overwrite the latest checkpoint
                                                                #   (If False, saves all checkpoints)

# LoRA Arguments
use_lora: bool = True                                           # Whether to use LoRA fine-tuning
lora_rank: int = 32                                             # Rank of LoRA weight matrix
lora_dropout: float = 0.0                                       # Dropout applied to LoRA weights
use_quantization: bool = False                                  # Whether to 4-bit quantize VLA for LoRA fine-tuning
                                                                #   => CAUTION: Reduces memory but hurts performance

# Tracking Parameters
wandb_entity: str = "pollen"          # Name of WandB entity
wandb_project: str = "openvla"        # Name of WandB project                         # Name of entity to log under
run_id_note: Optional[str] = None                               # Extra note for logging, Weights & Biases

# fmt: on


Current working directory: /home/ids/ext-5219/tokenizer/openvla-oft


/home/ids/ext-5219/miniconda3/envs/openvla-oft/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-01-26 10:48:23.491813: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-01-26 10:48:23.524530: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-01-26 10:48:23.524579: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026

Using PUSHT constants:
  NUM_ACTIONS_CHUNK = 1
  ACTION_DIM = 7
  PROPRIO_DIM = 8
  ACTION_PROPRIO_NORMALIZATION_TYPE = bounds_q99
If needed, manually set the correct constants in `prismatic/vla/constants.py`!


2026-01-26 10:48:27.740776: W tensorflow/core/common_runtime/gpu/gpu_device.cc:2348] TensorFlow was not built with CUDA kernel binaries compatible with compute capability 9.0. CUDA kernels will be jit-compiled from PTX, which could take 30 minutes or longer.


In [3]:
# FINE-TUNNING SCRIPT FOR OPENVLA MODEL PART 2
# PARAMETERS FOR MODEL 


print(f"Fine-tuning OpenVLA Model `{vla_path}` on `{dataset_name}`")

# [Validate] Ensure GPU Available & Set Device / Distributed Context
assert torch.cuda.is_available(), "Fine-tuning assumes at least one GPU is available!"
distributed_state = PartialState()
# torch.cuda.set_device(device_id := distributed_state.local_process_index)
device_id=0 #I will use only one GPU for the momment
device_id = distributed_state.local_process_index
torch.cuda.set_device(device_id)
torch.cuda.empty_cache()

# Configure Unique Experiment ID & Log Directory
exp_id = (
    f"{vla_path.split('/')[-1]}+{dataset_name}"
    f"+b{batch_size * grad_accumulation_steps}"
    f"+lr-{learning_rate}"
)
if use_lora:
    exp_id += f"+lora-r{lora_rank}+dropout-{lora_dropout}"
if use_quantization:
    exp_id += "+q-4bit"
if run_id_note is not None:
    exp_id += f"--{run_id_note}"
if image_aug:
    exp_id += "--image_aug"

# Start =>> Build Directories
run_dir, adapter_dir = run_root_dir / exp_id, adapter_tmp_dir / exp_id
os.makedirs(run_dir, exist_ok=True)

# Quantization Config =>> only if LoRA fine-tuning
quantization_config = None
if use_quantization:
    assert use_lora, "Quantized training only supported for LoRA fine-tuning!"
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_quant_type="nf4"
    )

# Register OpenVLA model to HF Auto Classes (not needed if the model is on HF Hub)
AutoConfig.register("openvla", OpenVLAConfig)
AutoImageProcessor.register(OpenVLAConfig, PrismaticImageProcessor)
AutoProcessor.register(OpenVLAConfig, PrismaticProcessor)
AutoModelForVision2Seq.register(OpenVLAConfig, OpenVLAForActionPrediction)

# Load OpenVLA Processor and Model using HF AutoClasses
processor = AutoProcessor.from_pretrained(vla_path, trust_remote_code=True)
vla = AutoModelForVision2Seq.from_pretrained(
    vla_path,
    torch_dtype=torch.bfloat16,
    quantization_config=quantization_config,
    low_cpu_mem_usage=True,
    trust_remote_code=True,
)

# Device Placement =>> note that BitsAndBytes automatically handles for quantized training
if use_quantization:
    vla = prepare_model_for_kbit_training(vla)
else:
    vla = vla.to(device_id)

    # [LoRA] Wrap Model w/ PEFT `LoraConfig` =>> by default we set `target_modules=all-linear`
    if use_lora:
        lora_config = LoraConfig(
            r=lora_rank,
            lora_alpha=min(lora_rank, 16),
            lora_dropout=lora_dropout,
            target_modules="all-linear",
            init_lora_weights="gaussian",
        )
        vla = get_peft_model(vla, lora_config)
        vla.print_trainable_parameters()

    # Wrap VLA in PyTorch DDP Wrapper for Multi-GPU Training
    # vla = DDP(vla, device_ids=[device_id], find_unused_parameters=True, gradient_as_bucket_view=True)

    # Create Optimizer =>> note that we default to a simple constant learning rate!
    trainable_params = [param for param in vla.parameters() if param.requires_grad]
    optimizer = AdamW(trainable_params, lr=learning_rate)

    # Create Action Tokenizer
    action_tokenizer = ActionTokenizer(processor.tokenizer)

Fine-tuning OpenVLA Model `openvla/openvla-7b` on `columbia_cairlab_pusht_real`


KeyboardInterrupt: 

In [ ]:
# FINE-TUNNING SCRIPT FOR OPENVLA MODEL PART 3
# LOADING DATASET
# Create training and optional validation datasets

# Load Fine-tuning Dataset =>> note that we use an RLDS-formatted dataset following Open X-Embodiment by default.
#   =>> If you want to use a non-RLDS dataset (e.g., a standard PyTorch Dataset) see the following commented block.
#   =>> Note that our training code does not loop over epochs because the RLDS loader does this implicitly; if using
#       your own Dataset, make sure to add the appropriate logic to the training loop!
#
# ---
# from prismatic.vla.datasets import DummyDataset
#
# vla_dataset = DummyDataset(
#     action_tokenizer,
#     processor.tokenizer,
#     image_transform=processor.image_processor.apply_transform,
#     prompt_builder_fn=PurePromptBuilder if "v01" not in vla_path else VicunaV15ChatPromptBuilder,
# )
# ---
batch_transform = RLDSBatchTransform(
    action_tokenizer,
    processor.tokenizer,
    image_transform=processor.image_processor.apply_transform,
    prompt_builder_fn=PurePromptBuilder if "v01" not in vla_path else VicunaV15ChatPromptBuilder,
)
vla_dataset = RLDSDataset(
    data_root_dir,
    dataset_name,
    batch_transform,
    resize_resolution=tuple(vla.config.image_sizes),
    shuffle_buffer_size=shuffle_buffer_size,
    image_aug=image_aug,
)

# [Important] Save Dataset Statistics =>> used to de-normalize actions for inference!
if distributed_state.is_main_process:
    save_dataset_statistics(vla_dataset.dataset_statistics, run_dir)

# Create Collator and DataLoader
collator = PaddedCollatorForActionPrediction(
    processor.tokenizer.model_max_length, processor.tokenizer.pad_token_id, padding_side="right"
)
dataloader = DataLoader(
    vla_dataset,
    batch_size=batch_size,
    sampler=None,
    collate_fn=collator,
    num_workers=0,  # Important =>> Set to 0 if using RLDS; TFDS rolls its own parallelism!
)

# Initialize Logging =>> W&B
if distributed_state.is_main_process:
    wandb.init(entity=wandb_entity, project=wandb_project, name=f"ft+{exp_id}")

01/26 [10:41:26] INFO     | >> Load dataset info from                                           ]8;id=677530;file:///home/ids/ext-5219/miniconda3/envs/openvla-oft/lib/python3.10/site-packages/tensorflow_datasets/core/dataset_info.py\dataset_info.py]8;;\:]8;id=337126;file:///home/ids/ext-5219/miniconda3/envs/openvla-oft/lib/python3.10/site-packages/tensorflow_datasets/core/dataset_info.py#599\599]8;;\
                          /home/ids/ext-5219/tokenizer/test/columbia_cairlab_pusht_real/1.0.0                      

                 WARNING  | >> `FeatureConnector.dtype` is deprecated. Please change your code to use ]8;id=269003;file:///home/ids/ext-5219/miniconda3/envs/openvla-oft/lib/python3.10/site-packages/tensorflow_datasets/core/features/feature.py\feature.py]8;;\:]8;id=671551;file:///home/ids/ext-5219/miniconda3/envs/openvla-oft/lib/python3.10/site-packages/tensorflow_datasets/core/features/feature.py#67\67]8;;\
                          NumPy with the field `FeatureConnector.np_dtype` or use TensorFlow with the              
                          field `FeatureConnector.tf_dtype`.                                                       

                 WARNING  | >> `FeatureConnector.dtype` is deprecated. Please change your code to use ]8;id=938176;file:///home/ids/ext-5219/miniconda3/envs/openvla-oft/lib/python3.10/site-packages/tensorflow_datasets/core/features/feature.py\feature.py]8;;\:]8;id=95434;file:///home/ids/ext-5219/miniconda3/envs/openvla-oft/lib/python3.10/site-packages/tensorflow_datasets/core/features/feature.py#67\67]8;;\
                          NumPy with the field `FeatureConnector.np_dtype` or use TensorFlow with the              
                          field `FeatureConnector.tf_dtype`.                                                       

                 INFO     | >> Constructing tf.data.Dataset columbia_cairlab_pusht_real for    ]8;id=884664;file:///home/ids/ext-5219/miniconda3/envs/openvla-oft/lib/python3.10/site-packages/tensorflow_datasets/core/logging/logging_logger.py\logging_logger.py]8;;\:]8;id=743635;file:///home/ids/ext-5219/miniconda3/envs/openvla-oft/lib/python3.10/site-packages/tensorflow_datasets/core/logging/logging_logger.py#49\49]8;;\
                          split all, from                                                                          
                          /home/ids/ext-5219/tokenizer/test/columbia_cairlab_pusht_real/1.0.0                      

2026-01-26 10:41:26.896961: I tensorflow/core/grappler/optimizers/data/replicate_on_split.cc:32] Running replicate on split optimization


...

                 INFO     | >> Constructing tf.data.Dataset columbia_cairlab_pusht_real for    ]8;id=266604;file:///home/ids/ext-5219/miniconda3/envs/openvla-oft/lib/python3.10/site-packages/tensorflow_datasets/core/logging/logging_logger.py\logging_logger.py]8;;\:]8;id=292312;file:///home/ids/ext-5219/miniconda3/envs/openvla-oft/lib/python3.10/site-packages/tensorflow_datasets/core/logging/logging_logger.py#49\49]8;;\
                          split train, from                                                                        
                          /home/ids/ext-5219/tokenizer/test/columbia_cairlab_pusht_real/1.0.0                      

2026-01-26 10:41:27.603880: I tensorflow/core/grappler/optimizers/data/replicate_on_split.cc:32] Running replicate on split optimization


01/26 [10:41:28] INFO     | >> [*] Applying frame transforms on dataset...                           ]8;id=333586;file:///home/ids/ext-5219/tokenizer/openvla-oft/prismatic/vla/datasets/rlds/dataset.py\dataset.py]8;;\:]8;id=833285;file:///home/ids/ext-5219/tokenizer/openvla-oft/prismatic/vla/datasets/rlds/dataset.py#582\582]8;;\

01/26 [10:41:29] INFO     | >> [*] Saved dataset statistics file at path                          ]8;id=445313;file:///home/ids/ext-5219/tokenizer/openvla-oft/prismatic/vla/datasets/rlds/utils/data_utils.py\data_utils.py]8;;\:]8;id=201746;file:///home/ids/ext-5219/tokenizer/openvla-oft/prismatic/vla/datasets/rlds/utils/data_utils.py#284\284]8;;\
                          runs/openvla-7b+columbia_cairlab_pusht_real+b8+lr-0.0005+lora-r32+dropo                  
                          ut-0.0--image_aug/dataset_statistics.json                                                

wandb: Currently logged in as: cataclysme-apocalypse (pollen) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [ ]:
# FINE-TUNNING SCRIPT FOR OPENVLA OFT MODEL PART 4
#TRAINING LOOP  # ==================================================

# Deque to store recent train metrics (used for computing smoothened metrics for gradient accumulation)
recent_losses = deque(maxlen=grad_accumulation_steps)
recent_action_accuracies = deque(maxlen=grad_accumulation_steps)
recent_l1_losses = deque(maxlen=grad_accumulation_steps)

# Train!
with tqdm.tqdm(total=max_steps, leave=False) as progress:
    vla.train()
    optimizer.zero_grad()
    for batch_idx, batch in enumerate(dataloader):
        with torch.autocast("cuda", dtype=torch.bfloat16):
            output: CausalLMOutputWithPast = vla(
                input_ids=batch["input_ids"].to(device_id),
                attention_mask=batch["attention_mask"].to(device_id),
                pixel_values=batch["pixel_values"].to(torch.bfloat16).to(device_id),
                labels=batch["labels"],
            )
            loss = output.loss

        # Normalize loss to account for gradient accumulation
        normalized_loss = loss / grad_accumulation_steps

        # Backward pass
        normalized_loss.backward()

        # Compute Accuracy and L1 Loss for Logging
        action_logits = output.logits[:, vla.vision_backbone.featurizer.patch_embed.num_patches : -1]
        action_preds = action_logits.argmax(dim=2)
        action_gt = batch["labels"][:, 1:].to(action_preds.device)
        mask = action_gt > action_tokenizer.action_token_begin_idx

        # Compute Accuracy
        correct_preds = (action_preds == action_gt) & mask
        action_accuracy = correct_preds.sum().float() / mask.sum().float()

        # Compute L1 Loss on Predicted (Continuous) Actions
        continuous_actions_pred = torch.tensor(
            action_tokenizer.decode_token_ids_to_actions(action_preds[mask].cpu().numpy())
        )
        continuous_actions_gt = torch.tensor(
            action_tokenizer.decode_token_ids_to_actions(action_gt[mask].cpu().numpy())
        )
        action_l1_loss = torch.nn.functional.l1_loss(continuous_actions_pred, continuous_actions_gt)

        # Store recent train metrics
        recent_losses.append(loss.item())
        recent_action_accuracies.append(action_accuracy.item())
        recent_l1_losses.append(action_l1_loss.item())

        # Compute gradient step index
        gradient_step_idx = batch_idx // grad_accumulation_steps

        # Compute smoothened train metrics
        #   =>> Equal to current step metrics when not using gradient accumulation
        #   =>> Otherwise, equal to the average of metrics observed over micro-batches used for gradient accumulation
        smoothened_loss = sum(recent_losses) / len(recent_losses)
        smoothened_action_accuracy = sum(recent_action_accuracies) / len(recent_action_accuracies)
        smoothened_l1_loss = sum(recent_l1_losses) / len(recent_l1_losses)

        # Push Metrics to W&B (every 10 gradient steps)
        if distributed_state.is_main_process and gradient_step_idx % 10 == 0:
            wandb.log(
                {
                    "train_loss": smoothened_loss,
                    "action_accuracy": smoothened_action_accuracy,
                    "l1_loss": smoothened_l1_loss,
                },
                step=gradient_step_idx,
            )

        # Optimizer Step
        if (batch_idx + 1) % grad_accumulation_steps == 0:
            optimizer.step()
            optimizer.zero_grad()
            progress.update()

        # Save Model Checkpoint =>> by default, only keeps the latest checkpoint, continually overwriting it!
        if gradient_step_idx > 0 and gradient_step_idx % save_steps == 0:
            if distributed_state.is_main_process:
                print(f"Saving Model Checkpoint for Step {gradient_step_idx}")

                # If LoRA, we first save adapter weights, then merge into full model; otherwise, default save!
                save_dir = adapter_dir if use_lora else run_dir

                # Save Processor & Weights
                processor.save_pretrained(run_dir)
                vla.save_pretrained(save_dir)

            # Wait for processor and adapter weights to be saved by main process
            # dist.barrier()

            # Merge LoRA weights into model backbone for faster inference
            #   =>> Note that merging is slow and can be done post-hoc to speed up training
            if use_lora:
                base_vla = AutoModelForVision2Seq.from_pretrained(
                    vla_path, torch_dtype=torch.bfloat16, low_cpu_mem_usage=True, trust_remote_code=True
                )
                merged_vla = PeftModel.from_pretrained(base_vla, adapter_dir)
                merged_vla = merged_vla.merge_and_unload()
                if distributed_state.is_main_process:
                    if save_latest_checkpoint_only:
                        # Overwrite latest checkpoint
                        merged_vla.save_pretrained(run_dir)

                        print(f"Saved Model Checkpoint for Step {gradient_step_idx} at: {run_dir}")
                    else:
                        # Prepare to save checkpoint in new directory
                        checkpoint_dir = Path(str(run_dir) + f"--{gradient_step_idx}_chkpt")
                        os.makedirs(checkpoint_dir, exist_ok=True)

                        # Save dataset statistics to new directory
                        save_dataset_statistics(vla_dataset.dataset_statistics, checkpoint_dir)

                        # Save processor and model weights to new directory
                        processor.save_pretrained(checkpoint_dir)
                        merged_vla.save_pretrained(checkpoint_dir)

                        print(f"Saved Model Checkpoint for Step {gradient_step_idx} at: {checkpoint_dir}")

            # Block on Main Process Checkpointing
            # dist.barrier()

        # Stop training when max_steps is reached
        if gradient_step_idx == max_steps:
            print(f"Max step {max_steps} reached! Stopping training...")
            break

  0%| | 0/300 [00:00<?, ?itWARNING: All log messages before absl::InitializeLog() is called are written to STDERR
W0000 00:00:1769420490.915338 2302224 op_level_cost_estimator.cc:699] Error in PredictCost() for the op: op: "CropAndResize" attr { key: "T" value { type: DT_FLOAT } } attr { key: "extrapolation_value" value { f: 0 } } attr { key: "method" value { s: "bilinear" } } inputs { dtype: DT_FLOAT shape { dim { size: 1 } dim { size: 224 } dim { size: 224 } dim { size: 3 } } } inputs { dtype: DT_FLOAT shape { dim { size: -2 } dim { size: 4 } } } inputs { dtype: DT_INT32 shape { dim { size: -2 } } } inputs { dtype: DT_INT32 shape { dim { size: 2 } } } device { type: "CPU" vendor: "AuthenticAMD" model: "241" frequency: 2995 num_cores: 8 environment { key: "cpu_instruction_set" value: "AVX SSE, SSE2, SSE3, SSSE3, SSE4.1, SSE4.2" } environment { key: "eigen" value: "3.4.90" } l1_cache_size: 32768 l2_cache_size: 1048576 l3_cache_size: 67108864 memory_size: 268435456 } outputs { dtype: DT

--- DEBUG: traj_len 250 ---
--- DEBUG: future_action_window_size 0 ---
--- DEBUG: traj_CLUSTERID_AVANT [250] ---
--- DEBUG: traj_ACTION [250 1 7] ---
--- DEBUG: traj_CLUSTERID_APRES [250 1] ---
--- DEBUG: traj_len 232 ---
--- DEBUG: future_action_window_size 0 ---
--- DEBUG: traj_CLUSTERID_AVANT [232] ---
--- DEBUG: traj_ACTION [232 1 7] ---
--- DEBUG: traj_CLUSTERID_APRES [232 1] ---
--- DEBUG: traj_len 282 ---
--- DEBUG: future_action_window_size 0 ---
--- DEBUG: traj_CLUSTERID_AVANT [282] ---
--- DEBUG: traj_ACTION [282 1 7] ---
--- DEBUG: traj_CLUSTERID_APRES [282 1] ---
--- DEBUG: traj_len 222 ---
--- DEBUG: future_action_window_size 0 ---
--- DEBUG: traj_CLUSTERID_AVANT [222] ---
--- DEBUG: traj_ACTION [222 1 7] ---
--- DEBUG: traj_CLUSTERID_APRES [222 1] ---
--- DEBUG: traj_len 200 ---
--- DEBUG: future_action_window_size 0 ---
--- DEBUG: traj_CLUSTERID_AVANT [200] ---
--- DEBUG: traj_ACTION [200 1 7] ---
--- DEBUG: traj_CLUSTERID_APRES [200 1] ---
--- DEBUG: traj_len 230 ---
--

  0%| | 1/300 [00:09<47:10,

--- DEBUG: action shape (1, 7) ---
--- DEBUG: future action shape (0, 7) ---
--- DEBUG: string current action shape 7 ---
--- DEBUG: string future action shape 0 ---
--- DEBUG: string  action shape 7 ---
--- DEBUG: action shape (1, 7) ---
--- DEBUG: future action shape (0, 7) ---
--- DEBUG: string current action shape 7 ---
--- DEBUG: string future action shape 0 ---
--- DEBUG: string  action shape 7 ---
--- DEBUG: action shape (1, 7) ---
--- DEBUG: future action shape (0, 7) ---
--- DEBUG: string current action shape 7 ---
--- DEBUG: string future action shape 0 ---
--- DEBUG: string  action shape 7 ---
--- DEBUG: action shape (1, 7) ---
--- DEBUG: future action shape (0, 7) ---
--- DEBUG: string current action shape 7 ---
--- DEBUG: string future action shape 0 ---
--- DEBUG: string  action shape 7 ---
--- DEBUG: action shape (1, 7) ---
--- DEBUG: future action shape (0, 7) ---
--- DEBUG: string current action shape 7 ---
--- DEBUG: string future action shape 0 ---
--- DEBUG: string 

  1%| | 2/300 [00:09<20:41,

...

--- DEBUG: action shape (1, 7) ---
--- DEBUG: future action shape (0, 7) ---
--- DEBUG: string current action shape 7 ---
--- DEBUG: string future action shape 0 ---
--- DEBUG: string  action shape 7 ---
--- DEBUG: action shape (1, 7) ---
--- DEBUG: future action shape (0, 7) ---
--- DEBUG: string current action shape 7 ---
--- DEBUG: string future action shape 0 ---
--- DEBUG: string  action shape 7 ---
--- DEBUG: action shape (1, 7) ---
--- DEBUG: future action shape (0, 7) ---
--- DEBUG: string current action shape 7 ---
--- DEBUG: string future action shape 0 ---
--- DEBUG: string  action shape 7 ---
--- DEBUG: action shape (1, 7) ---
--- DEBUG: future action shape (0, 7) ---
--- DEBUG: string current action shape 7 ---
--- DEBUG: string future action shape 0 ---
--- DEBUG: string  action shape 7 ---
--- DEBUG: action shape (1, 7) ---
--- DEBUG: future action shape (0, 7) ---
--- DEBUG: string current action shape 7 ---
--- DEBUG: string future action shape 0 ---
--- DEBUG: string 

 29%|▎| 87/300 [01:03<02:16

--- DEBUG: action shape (1, 7) ---
--- DEBUG: future action shape (0, 7) ---
--- DEBUG: string current action shape 7 ---
--- DEBUG: string future action shape 0 ---
--- DEBUG: string  action shape 7 ---
--- DEBUG: action shape (1, 7) ---
--- DEBUG: future action shape (0, 7) ---
--- DEBUG: string current action shape 7 ---
--- DEBUG: string future action shape 0 ---
--- DEBUG: string  action shape 7 ---
--- DEBUG: action shape (1, 7) ---
--- DEBUG: future action shape (0, 7) ---
--- DEBUG: string current action shape 7 ---
--- DEBUG: string future action shape 0 ---
--- DEBUG: string  action shape 7 ---
--- DEBUG: action shape (1, 7) ---
--- DEBUG: future action shape (0, 7) ---
--- DEBUG: string current action shape 7 ---
--- DEBUG: string future action shape 0 ---
--- DEBUG: string  action shape 7 ---
--- DEBUG: action shape (1, 7) ---
--- DEBUG: future action shape (0, 7) ---
--- DEBUG: string current action shape 7 ---
--- DEBUG: string future action shape 0 ---
--- DEBUG: string 

KeyboardInterrupt: 

wandb: 
wandb: 🚀 View run ft+openvla-7b+columbia_cairlab_pusht_real+b8+lr-0.0005+lora-r32+dropout-0.0--image_aug at: https://wandb.ai/pollen/openvla/runs/fz6cauzw
wandb: Find logs at: wandb/run-20260126_104129-fz6cauzw/logs


# INFERENCE OPENVLA

In [8]:
import sys
import os

os.chdir("/home/ids/ext-5219/tokenizer/openvla-oft/")
print("Current working directory:", os.getcwd())

sys.argv.append("pusht")



from PIL import Image
import numpy as np
from pathlib import Path
from prismatic.vla.datasets import RLDSBatchTransform, RLDSDataset

from prismatic.vla.constants import NUM_ACTIONS_CHUNK, PROPRIO_DIM

from experiments.robot.openvla_utils import _load_dataset_stats

pretrained_checkpoint ="/home/ids/ext-5219/tokenizer/openvla-oft/runs/openvla-7b+columbia_cairlab_pusht_real+b8+lr-0.0005+lora-r32+dropout-0.0--image_aug/"
# Instantiate config


# Load OpenVLA policy and inputs processor
processor = AutoProcessor.from_pretrained(pretrained_checkpoint, trust_remote_code=True)
vla = AutoModelForVision2Seq.from_pretrained(
    pretrained_checkpoint, 
    attn_implementation="flash_attention_2",  # [Optional] Requires `flash_attn`
    torch_dtype=torch.bfloat16, 
    low_cpu_mem_usage=True, 
    trust_remote_code=True
).to("cuda:0")

_load_dataset_stats(vla, pretrained_checkpoint)

print("✓ Modèle chargé avec succès !")

# Load dataset via RLDSDataset (same as training pipeline)
data_root_dir: Path = Path("/home/ids/ext-5219/tokenizer/test")
dataset_name: str = "columbia_cairlab_pusht_real"

# Create batch transform
action_tokenizer_inf = ActionTokenizer(processor.tokenizer)
batch_transform_inf = RLDSBatchTransform(
    action_tokenizer_inf,
    processor.tokenizer,
    image_transform=processor.image_processor.apply_transform,
    prompt_builder_fn=PurePromptBuilder,
    use_wrist_image=False,
    use_proprio=False,
    use_subtrajectory=True,
)

# Create dataset (this properly handles the data structure)
inference_dataset = RLDSDataset(
    data_root_dir,
    dataset_name,
    batch_transform_inf,
    resize_resolution=tuple(vla.config.image_sizes),
    shuffle_buffer_size=100,
    image_aug=False,
    train=True,
)

# Get first sample
sample_iterator = iter(inference_dataset)
sample_dict = next(sample_iterator)

print(f"✓ Dataset loaded. Sample keys: {sample_dict.keys()}")

# Extract ground truth subtrajectory_id (check if it exists)

ground_truth_subtrajectory_id = sample_dict["actions"]


# Extract and prepare observation for inference
pixel_values = sample_dict["pixel_values"]

# Convert from tensor to numpy if needed
if hasattr(pixel_values, "numpy"):
    pixel_values = pixel_values.numpy()

# The processor may create multi-channel images (e.g., 6 channels for 2 images)
# Extract only the first 3 channels (primary image)
if pixel_values.shape[0] > 3:
    pixel_values = pixel_values[:3]

# Now pixel_values should be (3, H, W) - transpose to (H, W, 3) using numpy
image_np = np.transpose(pixel_values, (1, 2, 0))

# Denormalize from ImageNet normalization to [0, 255]
if image_np.dtype in [np.float32, np.float64]:
    imagenet_mean = np.array([0.485, 0.456, 0.406])
    imagenet_std = np.array([0.229, 0.224, 0.225])
    image_np = (image_np * imagenet_std[np.newaxis, np.newaxis, :]) + imagenet_mean[np.newaxis, np.newaxis, :]
    image_np = np.clip(image_np, 0, 1)
    image_np = (image_np * 255).astype(np.uint8)
else:
    image_np = image_np.astype(np.uint8)



# Grab image input & format prompt
image: Image.Image = Image.fromarray(image_np)
prompt = "In: What action should the robot take to {<INSTRUCTION>}?\nOut:"


print("✓ Observation préparée avec succès !")
print(f"Image shape: {image_np.shape}, dtype: {image_np.dtype}")

# Generate robot action chunk (sequence of future actions)
print("\n→ Starting inference ...")
# Predict Action (7-DoF; un-normalize for BridgeData V2)
inputs = processor(prompt, image).to("cuda:0", dtype=torch.bfloat16)
action = vla.predict_action(**inputs, unnorm_key="columbia_cairlab_pusht_real", do_sample=False)


gt_cluster =ground_truth_subtrajectory_id

# S'assurer que les deux sont des tableaux numpy "plats" pour la comparaison
gt_flat = np.array(gt_cluster).flatten()
pred_flat = np.array(action).flatten()

# Comparaison avec une tolérance pour les flottants
is_match = np.allclose(gt_flat, pred_flat, atol=1e-3)

print(f"\n{'='*60}")
print(f"INFERENCE RESULTS:")
print(f"{'='*60}")

match = "✓ CORRECT" if is_match else "✗ MISMATCH"

print(f"Ground Truth: {gt_flat}")
print(f"Predicted:    {pred_flat}")
print(f"Result:       {match}")



Current working directory: /home/ids/ext-5219/tokenizer/openvla-oft


Loading checkpoint sha


✓ Modèle chargé avec succès !


2026-01-26 11:47:59.095540: I tensorflow/core/grappler/optimizers/data/replicate_on_split.cc:32] Running replicate on split optimization


01/26 [11:47:59] INFO     | >> [*] Loading existing dataset statistics from                       ]8;id=729214;file:///home/ids/ext-5219/tokenizer/openvla-oft/prismatic/vla/datasets/rlds/utils/data_utils.py\data_utils.py]8;;\:]8;id=173331;file:///home/ids/ext-5219/tokenizer/openvla-oft/prismatic/vla/datasets/rlds/utils/data_utils.py#199\199]8;;\
                          /home/ids/ext-5219/tokenizer/test/columbia_cairlab_pusht_real/1.0.0/dat                  
                          aset_statistics_d6170bf2de88fd222da6c9a2203ee8e1f88e82227a970154e370e5e                  
                          137360b3e.json.                                                                          

...

                 INFO     | >> [*] Reads per Dataset: [1]                                            ]8;id=345100;file:///home/ids/ext-5219/tokenizer/openvla-oft/prismatic/vla/datasets/rlds/dataset.py\dataset.py]8;;\:]8;id=116111;file:///home/ids/ext-5219/tokenizer/openvla-oft/prismatic/vla/datasets/rlds/dataset.py#539\539]8;;\

                 INFO     | >> [*] Constructing datasets...                                          ]8;id=204485;file:///home/ids/ext-5219/tokenizer/openvla-oft/prismatic/vla/datasets/rlds/dataset.py\dataset.py]8;;\:]8;id=823192;file:///home/ids/ext-5219/tokenizer/openvla-oft/prismatic/vla/datasets/rlds/dataset.py#542\542]8;;\

2026-01-26 11:47:59.366324: I tensorflow/core/grappler/optimizers/data/replicate_on_split.cc:32] Running replicate on split optimization


                 INFO     | >> [*] Applying frame transforms on dataset...                           ]8;id=37115;file:///home/ids/ext-5219/tokenizer/openvla-oft/prismatic/vla/datasets/rlds/dataset.py\dataset.py]8;;\:]8;id=802429;file:///home/ids/ext-5219/tokenizer/openvla-oft/prismatic/vla/datasets/rlds/dataset.py#582\582]8;;\

--- DEBUG: traj_len 195 ---
--- DEBUG: future_action_window_size 0 ---
--- DEBUG: traj_CLUSTERID_AVANT [195] ---
--- DEBUG: traj_ACTION [195 1 7] ---
--- DEBUG: traj_CLUSTERID_APRES [195 1] ---
--- DEBUG: traj_len 129 ---
--- DEBUG: future_action_window_size 0 ---
--- DEBUG: traj_CLUSTERID_AVANT [129] ---
--- DEBUG: traj_ACTION [129 1 7] ---
--- DEBUG: traj_CLUSTERID_APRES [129 1] ---
--- DEBUG: traj_len 166 ---
--- DEBUG: future_action_window_size 0 ---
--- DEBUG: traj_CLUSTERID_AVANT [166] ---
--- DEBUG: traj_ACTION [166 1 7] ---
--- DEBUG: traj_CLUSTERID_APRES [166 1] ---
--- DEBUG: traj_len 271 ---
--- DEBUG: future_action_window_size 0 ---
--- DEBUG: traj_CLUSTERID_AVANT [271] ---
--- DEBUG: traj_ACTION [271 1 7] ---
--- DEBUG: traj_CLUSTERID_APRES [271 1] ---
--- DEBUG: action shape (1, 7) ---
--- DEBUG: future action shape (0, 7) ---
--- DEBUG: string current action shape 7 ---
--- DEBUG: string future action shape 0 ---
--- DEBUG: string  action shape 7 ---
dimension cluster: (